# Brand → dim_mall Mapping Pipeline
Maps branch names from the 3-Day Sale Schedule to `branch_id` in `dim_mall`.

**Input files:**
- `dim_mall_0415.csv` — master mall dimension (80 branches)
- `YEAR_2026_3-Day_Sale_Schedule.xlsx` — sale schedule (743 rows, 104 unique branch names)

**Output:** `sale_schedule_mapped.csv` with `branch_id` resolved for each row

## Step 1 — Load files

In [44]:
import pandas as pd

# --- File paths (update if needed) ---
DIM_MALL_PATH   = 'dim_mall_0415.csv'
SALE_SCHED_PATH = '2026_3ds_sched.xlsx'
OUTPUT_PATH     = '3ds_schedule_mapped.csv'

df_mall = pd.read_csv(DIM_MALL_PATH)
df_sale = pd.read_excel(SALE_SCHED_PATH)

print(f'dim_mall      : {df_mall.shape[0]} rows, columns: {df_mall.columns.tolist()}')
print(f'sale schedule : {df_sale.shape[0]} rows, columns: {df_sale.columns.tolist()}')
df_sale.head(3)

dim_mall      : 80 rows, columns: ['branch_id', 'branch_name', 'mall_type', 'longitude', 'latitude', 'address', 'barangay_code', 'barangay_name', 'city_code', 'city_name', 'province_code', 'province_name', 'region_code', 'region_name', 'population', 'opening_date', 'age_in_months']
sale schedule : 743 rows, columns: ['branch_name', 'year', 'period', 'sale_date', 'access_type', 'sale_duration']


,branch_name,year,period,sale_date,access_type,sale_duration
0,North EDSA,2026,H1,2026-02-11,early access,5ds
1,North EDSA,2026,H1,2026-02-12,early access,5ds
2,North EDSA,2026,H1,2026-02-13,regular,5ds


## Step 2 — Normalize branch names

`dim_mall.branch_name` uses the format `"SM Store <Name>"`.  
Strip that prefix and whitespace from both sides so names are comparable.

In [45]:
df_mall['branch_name_clean'] = (
    df_mall['branch_name']
    .str.replace('SM Store ', '', regex=False)
    .str.strip()
)

df_sale['branch_name_clean'] = df_sale['branch_name'].str.strip()

print('dim_mall unique clean names:', df_mall['branch_name_clean'].nunique())
print('sale schedule unique names :', df_sale['branch_name_clean'].nunique())

dim_mall unique clean names: 80
sale schedule unique names : 104


## Step 3 — First-pass merge (exact match)

Left-join the sale schedule onto dim_mall using the cleaned name.  
Rows that don't match will have `branch_id = NaN`.

In [46]:
df_merged = df_sale.merge(
    df_mall[['branch_name_clean', 'branch_id', 'branch_name', 'city_name', 'region_name']],
    on='branch_name_clean',
    how='left',
    suffixes=('', '_dim')
)

matched   = df_merged['branch_id'].notna().sum()
unmatched = df_merged['branch_id'].isna().sum()
print(f'Matched   : {matched:>4} rows ({matched/len(df_merged)*100:.1f}%)')
print(f'Unmatched : {unmatched:>4} rows ({unmatched/len(df_merged)*100:.1f}%)')

Matched   :  538 rows (72.4%)
Unmatched :  205 rows (27.6%)


## Step 4 — Identify unmatched branch names

In [47]:
unmatched_names = sorted(
    df_merged.loc[df_merged['branch_id'].isna(), 'branch_name_clean'].unique()
)

print(f'{len(unmatched_names)} unmatched names:')
for name in unmatched_names:
    print(f'  "{name}"')

37 unmatched names:
  "CDO Uptown"
  "CHAIN PROMO"
  "Center - Angono"
  "Center - Imus"
  "Center - Lemery"
  "Center - Muntinlupa"
  "Center - Ormoc"
  "Center - Pasig"
  "Center - Pulilan"
  "Center - Sagandaan"
  "Center - Tuguegarao Downtown"
  "Center Shaw"
  "Center – Las Pinas"
  "Center – San Pedro"
  "JMall"
  "Lanang Premier"
  "MOA Square"
  "Mall of Asia"
  "Mindpro"
  "North EDSA"
  "Olongapo Downtown"
  "Roxas"
  "S'Maison"
  "SM Center Dagupan"
  "SM Center Imus"
  "SM Center Las Pinas"
  "SM Center Pasig"
  "SM Center Pulilan"
  "SM Center San Pedro"
  "SM Center Sangandaan"
  "SM Center Shaw"
  "SM Center Tuguegarao Downtown"
  "San Fernando Downtown"
  "Santa Rosa"
  "Sta. Rosa"
  "Sto. Tomas"
  "The Podium"


## Step 5 — Manual mapping dictionary

Three categories:
1. **Resolvable** — can be confidently mapped to a `dim_mall` branch
2. **Needs confirmation** — plausible match but verify with the business team
3. **No dim_mall entry** — these branches do not exist in dim_mall; rows will remain unmatched

**CHAIN PROMO** is not a branch — flag and exclude from the spatial join.

In [48]:
# Values of None = no match in dim_mall; row kept in output but branch_id stays NaN.
# Update any '# VERIFY' entries after confirming with the business team.

MANUAL_MAP = {
    # --- Resolvable: case / spacing / alias differences ---
    'North EDSA'           : 'North Edsa',
    'Mall of Asia'         : 'Mall Of Asia',
    'MOA Square'           : 'Mall Of Asia',          # same complex, different wing
    'Lanang Premier'       : 'Lanang',
    'Mindpro'              : 'Mindpro Zamboanga',
    'Roxas'                : 'Roxas City',
    'Santa Rosa'           : 'Sta Rosa',
    'Sta. Rosa'            : 'Sta Rosa',
    'Sto. Tomas'           : 'Sto Tomas',
    'Olongapo Downtown'    : 'Olongapo Central',
    'JMall'                : 'J Mall',
    'CDO Uptown'           : 'Cagayan De Oro',        # only one CDO branch in dim_mall
    'San Fernando Downtown': 'San Fernando',

    # --- Needs confirmation (VERIFY with business team before go-live) ---
    'Center - Imus'        : 'Bacoor',                # VERIFY
    'SM Center Imus'       : 'Bacoor',                # VERIFY
    'Center - Pasig'       : 'East Ortigas',          # VERIFY
    'SM Center Pasig'      : 'East Ortigas',          # VERIFY

    # --- No dim_mall entry (leave as NaN in output) ---
    'Center - Pulilan'               : None,
    'SM Center Pulilan'              : None,
    'Center - Sagandaan'             : None,
    'SM Center Sangandaan'           : None,
    'Center - Tuguegarao Downtown'   : None,
    'SM Center Tuguegarao Downtown'  : None,
    'Center Shaw'                    : None,
    'SM Center Shaw'                 : None,
    'Center \u2013 Las Pinas'        : None,
    'SM Center Las Pinas'            : None,
    'Center \u2013 San Pedro'        : None,
    'SM Center San Pedro'            : None,
    'Center - Angono'                : None,
    'Center - Lemery'                : None,
    'Center - Muntinlupa'            : None,
    'Center - Ormoc'                 : None,
    'SM Center Dagupan'              : None,
    '\u2018S\u2019Maison'            : None,
    "S\u2019Maison"                  : None,
    'The Podium'                     : None,

    # --- Special: chain-wide promo, not a branch ---
    'CHAIN PROMO'                    : '__CHAIN_PROMO__',
}

print(f'Manual map entries : {len(MANUAL_MAP)}')

Manual map entries : 38


## Step 6 — Apply mapping & re-merge

In [49]:
# Apply the manual map to produce a resolved name column
df_sale['branch_name_mapped'] = df_sale['branch_name_clean'].map(
    lambda x: MANUAL_MAP.get(x, x)  # fall back to cleaned name if not in map
)

# Flag CHAIN PROMO rows before the merge
df_sale['is_chain_promo'] = df_sale['branch_name_mapped'] == '__CHAIN_PROMO__'

# Re-merge using the mapped name
dim_cols = ['branch_name_clean', 'branch_id', 'branch_name']

df_final = df_sale.merge(
    df_mall[dim_cols].rename(columns={'branch_name': 'branch_name_dim'}),
    left_on='branch_name_mapped',
    right_on='branch_name_clean',
    how='left',
    suffixes=('', '_dim')
)

# Clean up: drop the sale schedule's branch_name and the duplicate clean name column
df_final.drop(columns=['branch_name', 'branch_name_clean_dim'], errors='ignore', inplace=True)

# Rename the dim_mall branch_name to branch_name for output
df_final.rename(columns={'branch_name_dim': 'branch_name'}, inplace=True)

matched   = df_final['branch_id'].notna().sum()
unmatched = df_final['branch_id'].isna().sum()
print(f'Total rows        : {len(df_final)}')
print(f'Matched (branch_id resolved) : {matched} ({matched/len(df_final)*100:.1f}%)')
print(f'Unmatched (branch_id = NaN)  : {unmatched} ({unmatched/len(df_final)*100:.1f}%)')

Total rows        : 743
Matched (branch_id resolved) : 652 (87.8%)
Unmatched (branch_id = NaN)  : 91 (12.2%)


## Step 7 — Validate & inspect residuals

In [50]:
# --- 7a. Summary by match status ---
summary = (
    df_final
    .assign(match_status=lambda d: d['branch_id'].notna().map({True: 'matched', False: 'unmatched'}))
    .groupby(['branch_name', 'match_status'])
    .size()
    .reset_index(name='row_count')
    .sort_values(['match_status', 'branch_name'])
)
print('=== Match summary by branch ===')
print(summary.to_string(index=False))

=== Match summary by branch ===
                branch_name match_status  row_count
              SM Store Aura      matched          8
      SM Store BF Paranaque      matched          8
           SM Store Bacolod      matched          8
            SM Store Bacoor      matched         14
            SM Store Baguio      matched          8
           SM Store Baliwag      matched          8
            SM Store Bataan      matched          8
          SM Store Batangas      matched          8
           SM Store Bicutan      matched          8
            SM Store Butuan      matched          8
      SM Store CDO Downtown      matched          8
        SM Store Cabanatuan      matched          8
    SM Store Cagayan De Oro      matched          8
           SM Store Calamba      matched          8
          SM Store Caloocan      matched          8
           SM Store Cauayan      matched          8
              SM Store Cebu      matched          8
             SM Store Clark     

In [51]:
# --- 7b. Unresolved rows (excludes CHAIN PROMO which is expected) ---
truly_unresolved = df_final[
    df_final['branch_id'].isna() & ~df_final['is_chain_promo']
][['branch_name', 'branch_name_clean', 'branch_name_mapped', 'sale_date']].drop_duplicates('branch_name')

print(f'Truly unresolved branches (not in dim_mall): {len(truly_unresolved)}')
print(truly_unresolved.to_string(index=False))

Truly unresolved branches (not in dim_mall): 1
branch_name branch_name_clean branch_name_mapped  sale_date
        NaN   Center - Angono                NaN 2026-02-13


In [52]:
# --- 7c. Quick sanity check: no duplicated branch_id per sale_date per branch ---
mask = df_final['branch_id'].notna()
dupes = df_final.duplicated(subset=['branch_id', 'sale_date', 'access_type'], keep=False) & mask

if dupes.any():
    print('WARNING: duplicate (branch_id, sale_date, access_type) rows found:')
    print(df_final[dupes][['branch_name', 'branch_id', 'sale_date', 'access_type']])
else:
    print('OK: no duplicate (branch_id, sale_date, access_type) rows')

               branch_name  branch_id  sale_date   access_type
214  SM Store Mall Of Asia      245.0 2026-04-29  early access
215  SM Store Mall Of Asia      245.0 2026-04-30  early access
216  SM Store Mall Of Asia      245.0 2026-05-01       regular
217  SM Store Mall Of Asia      245.0 2026-05-02       regular
218  SM Store Mall Of Asia      245.0 2026-05-03       regular
219  SM Store Mall Of Asia      245.0 2026-04-29  early access
220  SM Store Mall Of Asia      245.0 2026-04-30  early access
221  SM Store Mall Of Asia      245.0 2026-05-01       regular
222  SM Store Mall Of Asia      245.0 2026-05-02       regular
223  SM Store Mall Of Asia      245.0 2026-05-03       regular
542  SM Store Mall Of Asia      245.0 2026-09-16  early access
543  SM Store Mall Of Asia      245.0 2026-09-17  early access
544  SM Store Mall Of Asia      245.0 2026-09-18       regular
545  SM Store Mall Of Asia      245.0 2026-09-19       regular
546  SM Store Mall Of Asia      245.0 2026-09-20       

## Step 8 — Export mapped output

In [53]:
# Reorder columns for clarity
output_cols = [
    'branch_id',
    'branch_name',
    'branch_name_mapped',
    'year',
    'period',
    'sale_date',
    'access_type',
    'sale_duration',
]

# Keep only columns that exist in the dataframe
output_cols = [c for c in output_cols if c in df_final.columns]

df_output = df_final[output_cols].copy()
df_output['branch_id'] = df_output['branch_id'].astype('Int64')  # nullable integer

df_output.to_csv(OUTPUT_PATH, index=False)
print(f'Exported {len(df_output)} rows to {OUTPUT_PATH}')
df_output.head(5)

Exported 743 rows to 3ds_schedule_mapped.csv


,branch_id,branch_name,branch_name_mapped,year,period,sale_date,access_type,sale_duration
0,6,SM Store North Edsa,North Edsa,2026,H1,2026-02-11,early access,5ds
1,6,SM Store North Edsa,North Edsa,2026,H1,2026-02-12,early access,5ds
2,6,SM Store North Edsa,North Edsa,2026,H1,2026-02-13,regular,5ds
3,6,SM Store North Edsa,North Edsa,2026,H1,2026-02-14,regular,5ds
4,6,SM Store North Edsa,North Edsa,2026,H1,2026-02-15,regular,5ds


---
## Appendix — Branches still missing from dim_mall

The branches below appear in the sale schedule but have **no matching entry in dim_mall**.  
Coordinate with the data team to either add them to dim_mall or confirm they should be excluded.

| Sale schedule name | Notes |
|---|---|
| Center - Angono | SM Center, not in dim_mall |
| Center - Lemery | SM Center, not in dim_mall |
| Center - Muntinlupa | SM Center, not in dim_mall |
| Center - Ormoc | SM Center, not in dim_mall |
| Center - Pulilan | SM Center, not in dim_mall |
| Center - Sagandaan | SM Center, not in dim_mall |
| Center - Tuguegarao Downtown | SM Center, not in dim_mall |
| Center Shaw | SM Center, not in dim_mall |
| Center – Las Pinas | SM Center, not in dim_mall |
| Center – San Pedro | SM Center, not in dim_mall |
| SM Center Dagupan | Not in dim_mall |
| SM Center Las Pinas | Duplicate of Center – Las Pinas |
| SM Center Pulilan | Duplicate of Center - Pulilan |
| SM Center San Pedro | Duplicate of Center – San Pedro |
| SM Center Sangandaan | Duplicate of Center - Sagandaan |
| SM Center Shaw | Duplicate of Center Shaw |
| SM Center Tuguegarao Downtown | Duplicate of Center - Tuguegarao Downtown |
| S'Maison | Not in dim_mall |
| The Podium | Not in dim_mall |
| CHAIN PROMO | Chain-wide promo — not a branch; flagged via `is_chain_promo` |

**Entries needing business confirmation before use in analysis:**

| Sale schedule name | Mapped to | Confidence |
|---|---|---|
| Center - Imus / SM Center Imus | Bacoor | Medium — verify location |
| Center - Pasig / SM Center Pasig | East Ortigas | Medium — verify location |